In [1]:
from scipy.optimize import linprog

In [2]:
delay_probability=0.82
budget=20000
options={
    "Air Freight":{
        "cost":15000,
        "delay_days":3
    },
    "Secondary Supplier":{
        "cost":16500,
        "delay_days":5
    },
    "Delay Product Launch":{
        "cost":4000,
        "delay_days":14
    }
}

In [3]:
for action, values in options.items():
    score = values["cost"] + (values["delay_days"] * 1000)
    
    print(action, "→", score)

Air Freight → 18000
Secondary Supplier → 21500
Delay Product Launch → 18000


In [4]:
feasible_options = {
    action: values
    for action, values in options.items()
    if values["cost"] <= budget
}

feasible_options

{'Air Freight': {'cost': 15000, 'delay_days': 3},
 'Secondary Supplier': {'cost': 16500, 'delay_days': 5},
 'Delay Product Launch': {'cost': 4000, 'delay_days': 14}}

In [5]:
best_action = min(
    feasible_options,
    key=lambda action:
        feasible_options[action]["cost"]
        + feasible_options[action]["delay_days"] * 1000
)

best_action

'Air Freight'

In [6]:
import numpy as np
import pandas as pd

In [7]:
shipment={
    "shipment_id": "SHP01024",
    "order_quantity": 1200,
    "inventory_level": 800,
    "supplier_capacity": 5000,
    "shipping_cost": 12000,
    "delay_probability": 0.82
}

In [8]:
shipment

{'shipment_id': 'SHP01024',
 'order_quantity': 1200,
 'inventory_level': 800,
 'supplier_capacity': 5000,
 'shipping_cost': 12000,
 'delay_probability': 0.82}

In [23]:
actions=pd.DataFrame({
    "action":[
        "Air Freight",
        "Secondary supplier",
        "Delay Product Launch"
    ],
    "cost":[
        15000,
        16500,
        4000
    ],
    "delay_days":[
        3,5,14
    ],
    "capacity":[
        2000,
        3000,
        1200
    ]
})

In [24]:
actions

,action,cost,delay_days,capacity
0,Air Freight,15000,3,2000
1,Secondary supplier,16500,5,3000
2,Delay Product Launch,4000,14,1200


In [25]:
delay_penalty=1000
actions["objective_cost"]=(
    actions["cost"]+
    actions["delay_days"]*delay_penalty
)

In [26]:
actions

,action,cost,delay_days,capacity,objective_cost
0,Air Freight,15000,3,2000,18000
1,Secondary supplier,16500,5,3000,21500
2,Delay Product Launch,4000,14,1200,18000


In [27]:
c=actions["objective_cost"].values

In [28]:
c

array([18000, 21500, 18000])

In [29]:
A_eq = np.array([
    [1, 1, 1]
])

b_eq = np.array([1])

In [30]:
budget=20000

In [31]:
A_ub = np.array([
    actions["cost"].values
])

b_ub = np.array([
    budget
])

In [32]:
bounds = [
    (0, 1),
    (0, 1),
    (0, 1)
]

In [33]:
result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

In [34]:
result.success

True

In [35]:
result.x

array([1., 0., 0.])

In [36]:
selected_index = np.argmax(result.x)

selected_action = actions.iloc[selected_index]["action"]

selected_action

'Air Freight'

In [37]:
print("Recommended Action:", selected_action)
print(
    "Estimated Cost: $",
    actions.iloc[selected_index]["cost"]
)
print(
    "Expected Delay:",
    actions.iloc[selected_index]["delay_days"],
    "days"
)

Recommended Action: Air Freight
Estimated Cost: $ 15000
Expected Delay: 3 days
